# EDA — Open-Set DGA Detection Dataset
Phân tích khám phá dữ liệu theo thứ tự:
1. Load & kiểm tra cơ bản
2. Thống kê cơ bản & Data Quality
3. Sanity Checks (zero-leakage verification)
4. Feature Engineering
5. Trực quan hóa phân phối features
6. Multivariate Analysis
7. TLD Distribution
8. Family Distribution
9. t-SNE Visualization
10. OOD Source Analysis

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import math
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

# Màu nhất quán cho 4 nhóm
GROUP_COLORS = {
    'Known Benign':  '#2563EB',
    'Known DGA':     '#EA580C',
    'Unknown DGA':   '#16A34A',
    'OOD':           '#9333EA'
}
GROUP_ORDER = ['Known Benign', 'Known DGA', 'Unknown DGA', 'OOD']
FIGURE_DIR = Path('../figures/EDA')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')
print(f'Figures -> {FIGURE_DIR.resolve()}')

Imports OK


In [6]:
from pathlib import Path

base_path = Path('../data/processed')

df_train          = pd.read_csv(base_path / 'known'          / 'train.csv')
df_val            = pd.read_csv(base_path / 'known'          / 'val.csv')
df_test_known     = pd.read_csv(base_path / 'known'          / 'test_known.csv')
df_unknown_family = pd.read_csv(base_path / 'unknown_family' / 'test_unknown_family.csv')
df_unknown_ood    = pd.read_csv(base_path / 'unknown_ood'    / 'test_unknown_ood.csv')

datasets = {
    'Train':          df_train,
    'Validation':     df_val,
    'Test Known':     df_test_known,
    'Unknown Family': df_unknown_family,
    'OOD':            df_unknown_ood
}

for name, df in datasets.items():
    print(f'\n===== {name} =====')
    print(f'  Rows: {df.shape[0]:,}  |  Cols: {df.shape[1]}')
    print(f'  Columns: {list(df.columns)}')
print('\nLoaded successfully!')

FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\known\\train.csv'

## 1. Thống kê cơ bản & Data Quality

In [ ]:
print('Dataset sizes:')
total = 0
for name, df in datasets.items():
    print(f'  {name:20s}: {df.shape[0]:>8,} rows')
    total += df.shape[0]
print(f'  {"TOTAL":20s}: {total:>8,} rows')

In [ ]:
print('Train label distribution:')
print(df_train['label'].value_counts())
print('\nTrain family distribution (top 10):')
print(df_train['family'].value_counts().head(10))

In [ ]:
print('Missing values check (Train):')
mv = df_train.isnull().sum()
print(mv[mv > 0] if mv.sum() > 0 else '  No missing values — OK')

print('\nMissing values check (all splits):')
for name, df in datasets.items():
    total_missing = df.isnull().sum().sum()
    print(f'  {name:20s}: {total_missing} missing')

### Biểu đồ phân bố label & split size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A — split sizes
split_names = list(datasets.keys())
split_sizes = [len(df) for df in datasets.values()]
colors_bar  = ['#2563EB', '#93C5FD', '#EA580C', '#16A34A', '#9333EA']
axes[0].bar(split_names, split_sizes, color=colors_bar, zorder=3)
for i, v in enumerate(split_sizes):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('So luong mau theo split', fontweight='bold')
axes[0].set_ylabel('So dong')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(axis='y', alpha=0.4)

# Panel B — label distribution in train
label_counts = df_train['label'].value_counts()
axes[1].pie(label_counts.values, labels=label_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=['#2563EB', '#EA580C'])
axes[1].set_title('Phan bo label trong Train set', fontweight='bold')

fig.suptitle('Dataset Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_01_split_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Sanity Checks — Zero Leakage Verification
> Kiểm tra không có domain nào bị rò rỉ giữa các tập train/test/unknown.

In [ ]:
train_domains   = set(df_train['domain'])
val_domains     = set(df_val['domain'])
test_domains    = set(df_test_known['domain'])
unknown_domains = set(df_unknown_family['domain'])
ood_domains     = set(df_unknown_ood['domain'])

checks = [
    ('Train ∩ Val',            train_domains & val_domains),
    ('Train ∩ Test Known',     train_domains & test_domains),
    ('Train ∩ Unknown Family', train_domains & unknown_domains),
    ('Train ∩ OOD',            train_domains & ood_domains),
    ('Test Known ∩ Unknown',   test_domains  & unknown_domains),
    ('Unknown Family ∩ OOD',   unknown_domains & ood_domains),
]

all_ok = True
for name, overlap in checks:
    status = 'OK' if len(overlap) == 0 else 'LEAK DETECTED'
    if len(overlap) > 0:
        all_ok = False
    print(f'  {name:35s}: {len(overlap):>5} overlap  [{status}]')

print()
if all_ok:
    print('ZERO DATA LEAKAGE confirmed — all splits are clean.')
else:
    print('WARNING: leakage detected! Check pipeline.')

# Kiểm tra family hold-out
train_families   = set(df_train['family'].unique())
unknown_families = set(df_unknown_family['family'].unique())
family_overlap   = train_families & unknown_families
print(f'\n  Family overlap (must be 0): {len(family_overlap)}')
print(f'  Hold-out families: {sorted(unknown_families)}')

## 3. Feature Engineering
Trích xuất các đặc trưng từ vựng (lexical features) để so sánh giữa 4 nhóm.

In [ ]:
vowels = set('aeiou')

def shannon_entropy(s):
    if not s: return 0.0
    counts = Counter(s)
    n = len(s)
    return -sum((c/n)*math.log2(c/n) for c in counts.values())

def bigram_diversity(s):
    s = s.lower()
    bigrams = [s[i:i+2] for i in range(len(s)-1)]
    return len(set(bigrams)) / len(bigrams) if bigrams else 0

def add_features(df):
    df = df.copy()
    clean = df['domain'].str.lower().str.strip()
    no_dot = clean.str.replace('.', '', regex=False)
    df['length']          = clean.apply(len)
    df['entropy']         = no_dot.apply(shannon_entropy)
    df['digit_ratio']     = no_dot.apply(lambda s: sum(c.isdigit() for c in s) / max(len(s),1))
    df['vowel_ratio']     = no_dot.apply(lambda s: sum(c in vowels for c in s) / max(len(s),1))
    df['bigram_diversity']= no_dot.apply(bigram_diversity)
    df['unique_char_ratio']= no_dot.apply(lambda s: len(set(s)) / max(len(s),1))
    df['tld']             = clean.apply(lambda s: s.split('.')[-1] if '.' in s else '')
    return df

# Áp dụng
df_benign         = add_features(df_test_known[df_test_known['label']=='benign'].copy())
df_known_dga      = add_features(df_test_known[df_test_known['label']=='dga'].copy())
df_unk_family     = add_features(df_unknown_family.copy())
df_ood_feat       = add_features(df_unknown_ood.copy())

df_benign['group']     = 'Known Benign'
df_known_dga['group']  = 'Known DGA'
df_unk_family['group'] = 'Unknown DGA'
df_ood_feat['group']   = 'OOD'

df_all = pd.concat([df_benign, df_known_dga, df_unk_family, df_ood_feat], ignore_index=True)
print(f'df_all shape: {df_all.shape}')
print(df_all['group'].value_counts())

## 4. Phân phối Features — KDE plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
features = [
    ('entropy',          'Shannon Entropy'),
    ('digit_ratio',      'Digit Ratio'),
    ('vowel_ratio',      'Vowel Ratio'),
    ('bigram_diversity', 'Bigram Diversity'),
    ('unique_char_ratio','Unique Char Ratio'),
    ('length',           'Domain Length'),
]

for ax, (feat, title) in zip(axes.flat, features):
    for group in GROUP_ORDER:
        subset = df_all[df_all['group'] == group]
        sns.kdeplot(subset[feat], ax=ax, label=group,
                    color=GROUP_COLORS[group], fill=True, alpha=0.25, linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(feat)
    ax.legend(fontsize=8)

fig.suptitle('Feature Distribution — 4 Groups (KDE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_02_kde_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, (feat, title) in zip(axes.flat, features):
    data_by_group = [df_all[df_all['group']==g][feat].dropna().values for g in GROUP_ORDER]
    bp = ax.boxplot(data_by_group, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='.', markersize=1, alpha=0.3))
    for patch, g in zip(bp['boxes'], GROUP_ORDER):
        patch.set_facecolor(GROUP_COLORS[g])
        patch.set_alpha(0.6)
    ax.set_xticks(range(1, len(GROUP_ORDER)+1))
    ax.set_xticklabels([g.replace(' ', '\n') for g in GROUP_ORDER], fontsize=8)
    ax.set_title(title, fontweight='bold')

fig.suptitle('Feature Distribution — Box Plot (4 Groups)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_03_boxplot_features.png', dpi=150, bbox_inches='tight')
plt.show()

### So sánh chi tiết Entropy: Benign vs DGA vs Unknown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: Benign vs Known DGA
ax = axes[0]
sns.kdeplot(df_benign['entropy'],    ax=ax, label='Known Benign',
            color=GROUP_COLORS['Known Benign'], fill=True, alpha=0.4)
sns.kdeplot(df_known_dga['entropy'], ax=ax, label='Known DGA',
            color=GROUP_COLORS['Known DGA'],    fill=True, alpha=0.4)
ax.set_title('Entropy: Known Benign vs Known DGA', fontweight='bold')
ax.set_xlabel('Shannon Entropy')
ax.legend()

# Panel B: Known DGA vs Unknown DGA
ax2 = axes[1]
sns.kdeplot(df_known_dga['entropy'],  ax=ax2, label='Known DGA',
            color=GROUP_COLORS['Known DGA'],   fill=True, alpha=0.4)
sns.kdeplot(df_unk_family['entropy'], ax=ax2, label='Unknown DGA',
            color=GROUP_COLORS['Unknown DGA'], fill=True, alpha=0.4)
ax2.set_title('Entropy: Known DGA vs Unknown DGA', fontweight='bold')
ax2.set_xlabel('Shannon Entropy')
ax2.legend()

fig.suptitle('Entropy — Phan tich chi tiet', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_04_entropy_detail.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Multivariate Analysis
Phân tích tương quan giữa các features và scatter plot đa chiều.

In [ ]:
feat_cols = ['entropy', 'digit_ratio', 'vowel_ratio', 'bigram_diversity',
             'unique_char_ratio', 'length']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap cho Known (train)
df_train_feat = add_features(df_train.copy())
corr_train = df_train_feat[feat_cols].corr()
sns.heatmap(corr_train, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[0], vmin=-1, vmax=1, linewidths=0.5)
axes[0].set_title('Correlation Matrix — Train (Known)', fontweight='bold')

# Heatmap cho OOD
corr_ood = df_ood_feat[feat_cols].corr()
sns.heatmap(corr_ood, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1], vmin=-1, vmax=1, linewidths=0.5)
axes[1].set_title('Correlation Matrix — OOD', fontweight='bold')

fig.suptitle('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Nhan xet: vowel_ratio va consonant_ratio thuong nghich chieu. digit_ratio it tuong quan voi entropy.')

In [ ]:
# Scatter plot: Entropy vs Digit Ratio — 4 groups
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample_n = 3000
rng = np.random.default_rng(42)

# Panel A: Entropy vs Digit Ratio
for group in GROUP_ORDER:
    sub = df_all[df_all['group']==group]
    idx = rng.choice(len(sub), size=min(sample_n, len(sub)), replace=False)
    axes[0].scatter(sub.iloc[idx]['entropy'], sub.iloc[idx]['digit_ratio'],
                    c=GROUP_COLORS[group], label=group, alpha=0.3, s=8)
axes[0].set_xlabel('Shannon Entropy')
axes[0].set_ylabel('Digit Ratio')
axes[0].set_title('Entropy vs Digit Ratio', fontweight='bold')
axes[0].legend(fontsize=9)

# Panel B: Entropy vs Vowel Ratio
for group in GROUP_ORDER:
    sub = df_all[df_all['group']==group]
    idx = rng.choice(len(sub), size=min(sample_n, len(sub)), replace=False)
    axes[1].scatter(sub.iloc[idx]['entropy'], sub.iloc[idx]['vowel_ratio'],
                    c=GROUP_COLORS[group], label=group, alpha=0.3, s=8)
axes[1].set_xlabel('Shannon Entropy')
axes[1].set_ylabel('Vowel Ratio')
axes[1].set_title('Entropy vs Vowel Ratio', fontweight='bold')
axes[1].legend(fontsize=9)

fig.suptitle('Scatter Plot — Multivariate (sample 3k per group)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_06_scatter_multivariate.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. t-SNE Visualization
Giảm chiều 6 features xuống 2D để thấy cấu trúc phân tách giữa 4 nhóm.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

TSNE_SAMPLE = 2000  # mỗi nhóm lấy 2000 mẫu
rng = np.random.default_rng(42)

parts = []
for group in GROUP_ORDER:
    sub = df_all[df_all['group']==group][feat_cols + ['group']].dropna()
    idx = rng.choice(len(sub), size=min(TSNE_SAMPLE, len(sub)), replace=False)
    parts.append(sub.iloc[idx])

df_tsne_input = pd.concat(parts, ignore_index=True)
X = StandardScaler().fit_transform(df_tsne_input[feat_cols].values)

print(f'Running t-SNE on {len(X)} samples...')
tsne = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000)
Z = tsne.fit_transform(X)
print('Done.')

fig, ax = plt.subplots(figsize=(9, 7))
for i, group in enumerate(GROUP_ORDER):
    mask = df_tsne_input['group'].values == group
    ax.scatter(Z[mask, 0], Z[mask, 1],
               c=GROUP_COLORS[group], label=group,
               alpha=0.4, s=10)

ax.set_title('t-SNE — 6 Lexical Features (2k samples/group)', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=10, markerscale=2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_07_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print('Nhan xet: Known Benign va Known DGA co xu huong phan tach ro. OOD overlap voi Benign nhieu hon — giai thich tai sao OOD detection kho.')

## 7. TLD Distribution

In [ ]:
def extract_tld(domain):
    parts = str(domain).split('.')
    return parts[-1] if len(parts) > 1 else ''

df_all['tld'] = df_all['domain'].apply(extract_tld)

for group in GROUP_ORDER:
    print(f'\nTop TLD — {group}')
    print(df_all[df_all['group']==group]['tld'].value_counts().head(5))

In [ ]:
top_tlds = df_all['tld'].value_counts().head(12).index.tolist()
df_tld   = df_all[df_all['tld'].isin(top_tlds)]

tld_pivot = df_tld.groupby(['tld', 'group']).size().unstack(fill_value=0)
tld_pivot = tld_pivot.loc[top_tlds, [g for g in GROUP_ORDER if g in tld_pivot.columns]]
tld_pivot_pct = tld_pivot.div(tld_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 6))
tld_pivot_pct.plot(kind='bar', stacked=True, ax=ax,
                   color=[GROUP_COLORS[g] for g in tld_pivot_pct.columns])
ax.set_xlabel('TLD', fontsize=11)
ax.set_ylabel('Phan tram (%)', fontsize=11)
ax.set_title('Phan bo TLD theo nhom (% stack)', fontsize=13, fontweight='bold')
ax.legend(title='Group', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_08_tld_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. DGA Family Distribution

In [ ]:
# Known families in train
family_counts = df_train[df_train['label']=='dga']['family'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Known families
family_counts.plot(kind='barh', ax=axes[0], color='#EA580C', alpha=0.8)
axes[0].set_title('Known DGA Family Distribution (Train)', fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# Panel B: Unknown (hold-out) families
unk_counts = df_unknown_family['family'].value_counts()
unk_counts.plot(kind='barh', ax=axes[1], color='#16A34A', alpha=0.8)
axes[1].set_title('Unknown (Hold-out) DGA Family Distribution', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

fig.suptitle('DGA Family Distribution — Known vs Unknown', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_09_family_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nKnown families: {len(family_counts)}')
print(f'Hold-out families: {len(unk_counts)}')
print(f'Hold-out families: {sorted(unk_counts.index.tolist())}')

## 9. OOD Source Analysis
Phân tích thành phần nguồn trong tập OOD — kiểm tra source diversity.

In [ ]:
src_counts = df_unknown_ood['source'].value_counts()
print('OOD source breakdown:')
for src, cnt in src_counts.items():
    pct = cnt / len(df_unknown_ood) * 100
    print(f'  {src:50s}: {cnt:>7,} ({pct:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
colors_src = plt.cm.Set2(np.linspace(0, 1, len(src_counts)))
ax.barh(src_counts.index, src_counts.values, color=colors_src, zorder=3)
for i, (src, cnt) in enumerate(src_counts.items()):
    ax.text(cnt + 200, i, f'{cnt:,}', va='center', fontsize=9)
ax.set_xlabel('So domain')
ax.set_title('OOD Source Breakdown', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_10_ood_source.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Domain Length Distribution
So sánh độ dài domain giữa 4 nhóm — feature quan trọng cho DGA detection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: KDE
for group in GROUP_ORDER:
    sub = df_all[df_all['group']==group]
    sns.kdeplot(sub['length'], ax=axes[0], label=group,
                color=GROUP_COLORS[group], fill=True, alpha=0.25, linewidth=1.5)
axes[0].set_title('Domain Length KDE', fontweight='bold')
axes[0].set_xlabel('Length (ky tu)')
axes[0].legend(fontsize=9)

# Panel B: Statistics table
stats_rows = []
for group in GROUP_ORDER:
    sub = df_all[df_all['group']==group]['length']
    stats_rows.append({
        'Group': group, 'Min': sub.min(), 'Max': sub.max(),
        'Mean': round(sub.mean(), 1), 'Median': sub.median(),
        'Std': round(sub.std(), 1)
    })
df_stats = pd.DataFrame(stats_rows)

axes[1].axis('off')
tbl = axes[1].table(cellText=df_stats.values, colLabels=df_stats.columns,
                    loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8)
axes[1].set_title('Domain Length Statistics', fontweight='bold', pad=20)

fig.suptitle('Domain Length Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'eda_11_domain_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary — Key Insights

| Insight | Ý nghĩa |
|---|---|
| **Known Benign vs Known DGA** tách biệt rõ theo entropy và vowel_ratio | 35 lexical features đủ mạnh để phân loại closed-set |
| **Known DGA vs Unknown DGA** phân phối tương đồng | Open-set detection khó hơn closed-set vì DGA families có cùng đặc trưng |
| **OOD overlap với Benign** trong t-SNE | Model có xu hướng nhầm OOD legitimate là benign — giải thích AUROC thấp trên unknown_ood |
| **tranco_tail** có OOD score thấp tương đương known benign | Benign overfit: model học phân phối Tranco, tranco_tail có cùng phân phối |
| **TLD `.ru`** chiếm tỷ lệ cao hơn trong Unknown DGA | Một số DGA families ưu tiên TLD cụ thể — feature hữu ích |
| **Source diversity** trong OOD pool | Dataset không bị dominated bởi một nguồn duy nhất — source capping hoạt động tốt |